## top control cell

In [3]:
import itertools
import traceback
from argparse import Namespace
import torch
import gc
from utils.final_run_config import infer_channels, resolve_window_horizon
from utils.final_run_executor_tmp import run_one_experiment, evaluate_one_experiment


In [4]:


# =========================================================
# FIXED TOP-LEVEL CONTROLS
# =========================================================
SEED = 42
EXPERIMENT_MODE = "increasing_lookback"   # "multivariate" | "increasing_lookback"
# EXPERIMENT_MODE = "multivariate"
ROOT_DIR_RUNS = "runs_debug"


# ---------------------------------------------------------
# Global run controls (kept outside YAML on purpose)
# ---------------------------------------------------------
BASE_CONFIG = Namespace(
    seed=SEED,
    verbose=False,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    stride=1,
    batch_size=32,
    # batch_size=8,    # for ecl/192
    ratios=(0.7, 0.15, 0.15),

    min_lr=1e-7,
    early_stop_patience=3,
    num_epochs=10,
)


# ---------------------------------------------------------
# This mode is the source of truth for (window, horizon)
# ---------------------------------------------------------
WINDOW, HORIZON = resolve_window_horizon(EXPERIMENT_MODE)


# ---------------------------------------------------------
# Full 12-config set for one fixed mode + one fixed seed
# tuple format: (data_name, model_name, token_style, window, horizon)
# ---------------------------------------------------------
datasets = ["ETTh1", "ECL"]
models = ["gru", "transformer", "mamba"]
token_styles = ["time", "variate"]

all_12_configs = [
    (data_name, model_name, token_style, WINDOW, HORIZON)
    for data_name, model_name, token_style in itertools.product(
        datasets,
        models,
        token_styles,
    )
]


# ---------------------------------------------------------
# Optional subset mode for debugging
# If empty / None -> run all 12
# ---------------------------------------------------------
test_configs = None

# Example:
# test_configs = [
#     ("ETTh1", "gru", "time", WINDOW, HORIZON),
#     ("ECL", "transformer", "variate", WINDOW, HORIZON),
# ]

test_configs = [
    ("ECL", "transformer", "time", WINDOW, HORIZON),
    ("ECL", "transformer", "variate", WINDOW, HORIZON),
]


configs_to_run = test_configs if test_configs else all_12_configs


# Your visual confirmation
print(f"EXPERIMENT_MODE = {EXPERIMENT_MODE}")
print(f"WINDOW, HORIZON = ({WINDOW}, {HORIZON})")
print(f"SEED = {SEED}")
print(f"BATCH_SIZE = {BASE_CONFIG.batch_size}")
print(f"ROOT_DIR_RUNS = {ROOT_DIR_RUNS}")
print(f"NUM CONFIGS = {len(configs_to_run)}")

EXPERIMENT_MODE = increasing_lookback
WINDOW, HORIZON = (192, 96)
SEED = 42
BATCH_SIZE = 32
ROOT_DIR_RUNS = runs_debug
NUM CONFIGS = 2


## loop runner

In [5]:


loop_results = []
loop_failures = []

total = len(configs_to_run)

for idx, (data_name, model_name, token_style, window, horizon) in enumerate(configs_to_run, start=1):
    cfg = Namespace(**vars(BASE_CONFIG))
    cfg.data_name = data_name
    cfg.model_name = model_name
    cfg.token_style = token_style
    cfg.window = window
    cfg.horizon = horizon
    cfg.channels = infer_channels(cfg.data_name)

    run_tag = (
        f"TRAIN [{idx}/{total}] "
        f"{cfg.data_name} | {cfg.model_name} | {cfg.token_style} | "
        f"L{cfg.window}_T{cfg.horizon} | BS={cfg.batch_size} | seed={cfg.seed}"
    )
    print(run_tag)

    try:
        result = run_one_experiment(
            base_cfg=cfg,
            experiment_mode=EXPERIMENT_MODE,

            # save_npz_splits=("val", "test"),
            # save_summary=True,

            # make_test_plot=True,
            # plot_num_random=3,
            # plot_seed=40,
            # plot_channel_idx=-1,

            root_dir_runs=ROOT_DIR_RUNS,
            verbose=False,
        )

        loop_results.append(result)

        print(
            f"       best_val={result['best_val_loss']:.6f} | "
            f"best_epoch={result['best_epoch']}"
        )
        print(f"       saved: {result['run_root']}")

    except Exception as e:
        loop_failures.append(
            {
                "idx": idx,
                "data_name": data_name,
                "model_name": model_name,
                "token_style": token_style,
                "window": window,
                "horizon": horizon,
                "seed": cfg.seed,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
        )

        print(f"       ERROR: {e}")

print("\n========== LOOP SUMMARY ==========")
print(f"Completed: {len(loop_results)}")
print(f"Failed   : {len(loop_failures)}")

TRAIN [1/2] ECL | transformer | time | L192_T96 | BS=32 | seed=42
       best_val=0.211571 | best_epoch=5
       saved: runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01
TRAIN [2/2] ECL | transformer | variate | L192_T96 | BS=32 | seed=42
       best_val=0.118741 | best_epoch=6
       saved: runs_debug/ECL/variate/transformer/L192_T96/seed_42/2026-03-14_21-06

========== LOOP SUMMARY ==========
Completed: 2
Failed   : 0


### 1) Evaluate only the first run

In [8]:
import gc
import torch
from utils.final_run_executor_tmp import evaluate_one_experiment

# choose one trained run from loop_results
train_res = loop_results[0]   # 0 = time, 1 = variate

eval_res = evaluate_one_experiment(
    cfg=train_res["config"],
    run_root=train_res["run_root"],
    save_npz_splits=None,
    save_summary=True,
    make_test_plot=False,
    include_val=False,
    verbose=False,
)

print("\nRESULT:")
print(eval_res["metrics"])
print(eval_res["test_dir"])
print(eval_res["test_npz_path"])
print(eval_res["test_plot_path"])


RESULT:
{'test': {'mse': 0.27084162973028353, 'mae': 0.3732276530124412, 'num_windows': 3659}}
runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test
None
None


### 2) Evaluate both runs, metrics only

In [ ]:
loop_results[1]["run_root"]

In [9]:
eval_results = []
eval_failures = []
total = len(loop_results)

for idx, train_res in enumerate(loop_results, start=1):
    cfg = train_res["config"]
    run_root = train_res["run_root"]

    print(
        f"[EVAL-METRICS {idx}/{total}] "
        f"{cfg.data_name} | {cfg.model_name} | {cfg.token_style} | "
        f"L{cfg.window}_T{cfg.horizon} | BS={cfg.batch_size} | seed={cfg.seed}"
    )

    try:
        eval_res = evaluate_one_experiment(
            cfg=cfg,
            run_root=run_root,
            save_npz_splits=None,
            save_summary=True,
            make_test_plot=False,
            include_val=False,
            verbose=False,
        )

        eval_results.append(eval_res)

        print(f"       test saved: {eval_res['test_dir']}")
        print(f"       test mse: {eval_res['metrics']['test']['mse']:.6f}")

    except Exception as e:
        eval_failures.append(
            {
                "config": cfg,
                "run_root": run_root,
                "error": str(e),
            }
        )
        print(f"       ERROR: {e}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\n========== EVAL METRICS SUMMARY ==========")
print(f"Completed: {len(eval_results)}")
print(f"Failed   : {len(eval_failures)}")

[EVAL-METRICS 1/2] ECL | transformer | time | L192_T96 | BS=32 | seed=42
       test saved: runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test
       test mse: 0.270842
[EVAL-METRICS 2/2] ECL | transformer | variate | L192_T96 | BS=32 | seed=42
       test saved: runs_debug/ECL/variate/transformer/L192_T96/seed_42/2026-03-14_21-06/test
       test mse: 0.133547

========== EVAL METRICS SUMMARY ==========
Completed: 2
Failed   : 0


### next step is NPZ

In [10]:
eval_results_npz = []
eval_failures_npz = []

total = len(loop_results)

for idx, train_res in enumerate(loop_results, start=1):
    cfg = train_res["config"]
    run_root = train_res["run_root"]

    print(
        f"[EVAL-NPZ {idx}/{total}] "
        f"{cfg.data_name} | {cfg.model_name} | {cfg.token_style} | "
        f"L{cfg.window}_T{cfg.horizon}"
    )

    try:
        eval_res = evaluate_one_experiment(
            cfg=cfg,
            run_root=run_root,
            save_npz_splits=("test",),
            save_summary=True,
            make_test_plot=False,
            include_val=False,
            verbose=True,
        )

        eval_results_npz.append(eval_res)

        print(f"       test mse: {eval_res['metrics']['test']['mse']:.6f}")
        print(f"       npz: {eval_res['test_npz_path']}")

    except Exception as e:
        eval_failures_npz.append(
            {
                "config": cfg,
                "run_root": run_root,
                "error": str(e),
            }
        )
        print(f"       ERROR: {e}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n========== EVAL NPZ SUMMARY ==========")
print(f"Completed: {len(eval_results_npz)}")
print(f"Failed   : {len(eval_failures_npz)}")


[EVAL-NPZ 1/2] ECL | transformer | time | L192_T96
EVAL  -> ECL | transformer | time | L192_T96 | seed=42
[evaluator_tmp] Processing split: test
[evaluator_tmp][metrics+npz] Y(3659, 96, 321), P(3659, 96, 321) | mse=0.270842 | mae=0.373228
[evaluator_tmp] Saved NPZ (Y,P only) -> runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/test_outputs.npz
[evaluator_tmp] Saved Summary JSON -> runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/post_training_metrics.json
       test mse: 0.270842
       npz: runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/test_outputs.npz
[EVAL-NPZ 2/2] ECL | transformer | variate | L192_T96
EVAL  -> ECL | transformer | variate | L192_T96 | seed=42
[evaluator_tmp] Processing split: test
[evaluator_tmp][metrics+npz] Y(3659, 96, 321), P(3659, 96, 321) | mse=0.133547 | mae=0.232097
[evaluator_tmp] Saved NPZ (Y,P only) -> runs_debug/ECL/variate/transformer/L192_T96/seed_42/2026-03-14_21-06/test/test_outputs

### then plots

In [11]:
eval_results_plot = []
eval_failures_plot = []

total = len(loop_results)

for idx, train_res in enumerate(loop_results, start=1):
    cfg = train_res["config"]
    run_root = train_res["run_root"]

    print(
        f"[EVAL-PLOT {idx}/{total}] "
        f"{cfg.data_name} | {cfg.model_name} | {cfg.token_style} | "
        f"L{cfg.window}_T{cfg.horizon}"
    )

    try:
        eval_res = evaluate_one_experiment(
            cfg=cfg,
            run_root=run_root,
            save_npz_splits=("test",),
            save_summary=True,
            make_test_plot=True,
            plot_num_random=3,
            plot_seed=40,
            plot_channel_idx=-1,
            include_val=False,
            verbose=True,
        )

        eval_results_plot.append(eval_res)

        print(f"       test mse : {eval_res['metrics']['test']['mse']:.6f}")
        print(f"       npz      : {eval_res['test_npz_path']}")
        print(f"       plot     : {eval_res['test_plot_path']}")

    except Exception as e:
        eval_failures_plot.append(
            {
                "config": cfg,
                "run_root": run_root,
                "error": str(e),
            }
        )
        print(f"       ERROR: {e}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n========== EVAL PLOT SUMMARY ==========")
print(f"Completed: {len(eval_results_plot)}")
print(f"Failed   : {len(eval_failures_plot)}")


[EVAL-PLOT 1/2] ECL | transformer | time | L192_T96
EVAL  -> ECL | transformer | time | L192_T96 | seed=42
[evaluator_tmp] Processing split: test
[evaluator_tmp][metrics+npz] Y(3659, 96, 321), P(3659, 96, 321) | mse=0.270842 | mae=0.373228
[evaluator_tmp] Saved NPZ (Y,P only) -> runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/test_outputs.npz
[evaluator_tmp] Saved Summary JSON -> runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/post_training_metrics.json
       test mse : 0.270842
       npz      : runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/test_outputs.npz
       plot     : runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-01/test/test_3_forecasts_ch-1.png
[EVAL-PLOT 2/2] ECL | transformer | variate | L192_T96
EVAL  -> ECL | transformer | variate | L192_T96 | seed=42
[evaluator_tmp] Processing split: test
[evaluator_tmp][metrics+npz] Y(3659, 96, 321), P(3659, 96, 321) | mse=0.133547 | mae=0.232097
[e

### across MODE

In [13]:
import gc
import traceback
from argparse import Namespace

import torch

from utils.final_run_config import infer_channels, resolve_window_horizon
from utils.final_run_executor_tmp import run_one_experiment, evaluate_one_experiment


SEED = 42
ROOT_DIR_RUNS = "runs_debug"

BASE_CONFIG = Namespace(
    seed=SEED,
    verbose=False,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    stride=1,
    batch_size=32,
    ratios=(0.7, 0.15, 0.15),
    min_lr=1e-7,
    early_stop_patience=3,
    num_epochs=10,
)

# 4 runs:
# - ECL transformer + ECL mamba
# - each across both modes
experiment_grid = [
    ("increasing_lookback", "ECL", "transformer", "time"),
    ("multivariate",        "ECL", "transformer", "time"),
    ("increasing_lookback", "ECL", "mamba",       "variate"),
    ("multivariate",        "ECL", "mamba",       "variate"),
]

train_results = []
train_failures = []
eval_results = []
eval_failures = []

total = len(experiment_grid)

for idx, (experiment_mode, data_name, model_name, token_style) in enumerate(experiment_grid, start=1):
    window, horizon = resolve_window_horizon(experiment_mode)

    cfg = Namespace(**vars(BASE_CONFIG))
    cfg.data_name = data_name
    cfg.model_name = model_name
    cfg.token_style = token_style
    cfg.window = window
    cfg.horizon = horizon
    cfg.channels = infer_channels(cfg.data_name)

    tag = (
        f"[{idx}/{total}] "
        f"mode={experiment_mode} | "
        f"{cfg.data_name} | {cfg.model_name} | {cfg.token_style} | "
        f"L{cfg.window}_T{cfg.horizon} | BS={cfg.batch_size} | seed={cfg.seed}"
    )
    print("\n" + "=" * 90)
    print(tag)

    # ---------------------------------------------------------
    # TRAIN
    # ---------------------------------------------------------
    try:
        train_res = run_one_experiment(
            base_cfg=cfg,
            experiment_mode=experiment_mode,
            root_dir_runs=ROOT_DIR_RUNS,
            verbose=False,
        )
        train_results.append(train_res)

        print(
            f"TRAIN OK  | best_val={train_res['best_val_loss']:.6f} | "
            f"best_epoch={train_res['best_epoch']}"
        )
        print(f"          saved: {train_res['run_root']}")

    except Exception as e:
        train_failures.append(
            {
                "idx": idx,
                "mode": experiment_mode,
                "data_name": data_name,
                "model_name": model_name,
                "token_style": token_style,
                "window": window,
                "horizon": horizon,
                "seed": cfg.seed,
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
        )
        print(f"TRAIN ERROR: {e}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        continue

    # ---------------------------------------------------------
    # EVAL (safe mode first: metrics only, test only)
    # ---------------------------------------------------------
    try:
        eval_res = evaluate_one_experiment(
            cfg=train_res["config"],
            run_root=train_res["run_root"],
            save_npz_splits=None,
            save_summary=True,
            make_test_plot=False,
            include_val=False,
            verbose=False,
        )
        eval_results.append(eval_res)

        print(f"EVAL  OK  | test mse={eval_res['metrics']['test']['mse']:.6f}")
        print(f"          test dir: {eval_res['test_dir']}")

    except Exception as e:
        eval_failures.append(
            {
                "idx": idx,
                "mode": experiment_mode,
                "data_name": data_name,
                "model_name": model_name,
                "token_style": token_style,
                "run_root": train_res["run_root"],
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
        )
        print(f"EVAL ERROR: {e}")

    # notebook-side cleanup too
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 90)
print("FINAL SUMMARY")
print(f"Train completed : {len(train_results)}")
print(f"Train failed    : {len(train_failures)}")
print(f"Eval completed  : {len(eval_results)}")
print(f"Eval failed     : {len(eval_failures)}")



[1/4] mode=increasing_lookback | ECL | transformer | time | L192_T96 | BS=32 | seed=42
TRAIN OK  | best_val=0.212016 | best_epoch=5
          saved: runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45
EVAL  OK  | test mse=0.268166
          test dir: runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/test

[2/4] mode=multivariate | ECL | transformer | time | L96_T96 | BS=32 | seed=42
TRAIN OK  | best_val=0.213499 | best_epoch=4
          saved: runs_debug/ECL/time/transformer/L96_T96/seed_42/2026-03-14_21-51
EVAL  OK  | test mse=0.267306
          test dir: runs_debug/ECL/time/transformer/L96_T96/seed_42/2026-03-14_21-51/test

[3/4] mode=increasing_lookback | ECL | mamba | variate | L192_T96 | BS=32 | seed=42
TRAIN OK  | best_val=0.122711 | best_epoch=7
          saved: runs_debug/ECL/variate/mamba/L192_T96/seed_42/2026-03-14_21-54
EVAL  OK  | test mse=0.136389
          test dir: runs_debug/ECL/variate/mamba/L192_T96/seed_42/2026-03-14_21-54/test

[4/4] m

In [16]:
len(train_results), len(eval_results)

(4, 4)

In [17]:
train_results[0]

{'config': Namespace(seed=42, verbose=False, device=device(type='cuda'), stride=1, batch_size=32, ratios=(0.7, 0.15, 0.15), min_lr=1e-07, early_stop_patience=3, num_epochs=10, data_name='ECL', model_name='transformer', token_style='time', window=192, horizon=96, channels=321, d_model=1024, num_layers=2, nhead=8, dim_feedforward=4096, activation='relu', norm_first=True, layer_norm_eps=1e-05, batch_first=True, pos_emb='sin', dropout=0.11099756667501104, pooling='last', temperature=1.0, lr=0.0001354163120034234, weight_decay=5.8386799552162966e-06, scheduler_factor=0.5653057659243387, scheduler_patience=3, final_yaml_path='configs/final_runs/increasing_lookback/ecl_transformer_time_L192_T96.yaml'),
 'yaml_path': 'configs/final_runs/increasing_lookback/ecl_transformer_time_L192_T96.yaml',
 'run_root': 'runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45',
 'train_dir': 'runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/train',
 'test_dir': 'runs_debug/ECL/time

In [18]:
eval_results[0]

{'config': Namespace(seed=42, verbose=False, device=device(type='cuda'), stride=1, batch_size=32, ratios=(0.7, 0.15, 0.15), min_lr=1e-07, early_stop_patience=3, num_epochs=10, data_name='ECL', model_name='transformer', token_style='time', window=192, horizon=96, channels=321, d_model=1024, num_layers=2, nhead=8, dim_feedforward=4096, activation='relu', norm_first=True, layer_norm_eps=1e-05, batch_first=True, pos_emb='sin', dropout=0.11099756667501104, pooling='last', temperature=1.0, lr=0.0001354163120034234, weight_decay=5.8386799552162966e-06, scheduler_factor=0.5653057659243387, scheduler_patience=3, final_yaml_path='configs/final_runs/increasing_lookback/ecl_transformer_time_L192_T96.yaml'),
 'run_root': 'runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45',
 'train_dir': 'runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/train',
 'test_dir': 'runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/test',
 'weights_path': 'runs_debug/ECL/time

### eval only on existing train_results with NPZ + plots

In [19]:
import gc
import torch

from utils.final_run_executor_tmp import evaluate_one_experiment

eval_results_plot = []
eval_failures_plot = []

total = len(train_results)

for idx, train_res in enumerate(train_results, start=1):
    cfg = train_res["config"]
    run_root = train_res["run_root"]

    print(
        f"[EVAL-PLOT {idx}/{total}] "
        f"{cfg.data_name} | {cfg.model_name} | {cfg.token_style} | "
        f"L{cfg.window}_T{cfg.horizon}"
    )

    try:
        eval_res = evaluate_one_experiment(
            cfg=cfg,
            run_root=run_root,
            save_npz_splits=("test",),
            save_summary=True,
            make_test_plot=True,
            plot_num_random=3,
            plot_seed=40,
            plot_channel_idx=-1,
            include_val=False,
            verbose=False,
        )

        eval_results_plot.append(eval_res)

        print(f"       test mse : {eval_res['metrics']['test']['mse']:.6f}")
        print(f"       npz      : {eval_res['test_npz_path']}")
        print(f"       plot     : {eval_res['test_plot_path']}")

    except Exception as e:
        eval_failures_plot.append(
            {
                "config": cfg,
                "run_root": run_root,
                "error": str(e),
            }
        )
        print(f"       ERROR: {e}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n========== EVAL PLOT SUMMARY ==========")
print(f"Completed: {len(eval_results_plot)}")
print(f"Failed   : {len(eval_failures_plot)}")


[EVAL-PLOT 1/4] ECL | transformer | time | L192_T96
       test mse : 0.268166
       npz      : runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/test/test_outputs.npz
       plot     : runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/test/test_3_forecasts_ch-1.png
[EVAL-PLOT 2/4] ECL | transformer | time | L96_T96
       test mse : 0.267306
       npz      : runs_debug/ECL/time/transformer/L96_T96/seed_42/2026-03-14_21-51/test/test_outputs.npz
       plot     : runs_debug/ECL/time/transformer/L96_T96/seed_42/2026-03-14_21-51/test/test_3_forecasts_ch-1.png
[EVAL-PLOT 3/4] ECL | mamba | variate | L192_T96
       test mse : 0.136389
       npz      : runs_debug/ECL/variate/mamba/L192_T96/seed_42/2026-03-14_21-54/test/test_outputs.npz
       plot     : runs_debug/ECL/variate/mamba/L192_T96/seed_42/2026-03-14_21-54/test/test_3_forecasts_ch-1.png
[EVAL-PLOT 4/4] ECL | mamba | variate | L96_T96
       test mse : 0.149846
       npz      : runs_debug/ECL/vari

### use helper now for your current train_results

In [20]:
import importlib
import utils.experiment_helpers_tmp as experiment_helpers_tmp

importlib.reload(experiment_helpers_tmp)

from utils.experiment_helpers_tmp import (
    build_base_config,
    make_experiment_grid,
    run_train_and_eval_loop,
    run_eval_only_loop,
)

print("Helpers reloaded.")


Helpers reloaded.


In [22]:
from utils.experiment_helpers_tmp import run_eval_only_loop

eval_plot_bundle = run_eval_only_loop(
    train_results=train_results,
    save_npz_splits=("test",),
    make_test_plot=True,
    plot_num_random=3,
    plot_seed=40,
    plot_channel_idx=-1,
    include_val=False,
    verbose=False,
)


[EVAL 1/4] ECL | transformer | time | L192_T96
       test mse : 0.268166
       npz      : runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/test/test_outputs.npz
       plot     : runs_debug/ECL/time/transformer/L192_T96/seed_42/2026-03-14_21-45/test/test_3_forecasts_ch-1_v2.png
[EVAL 2/4] ECL | transformer | time | L96_T96
       test mse : 0.267306
       npz      : runs_debug/ECL/time/transformer/L96_T96/seed_42/2026-03-14_21-51/test/test_outputs.npz
       plot     : runs_debug/ECL/time/transformer/L96_T96/seed_42/2026-03-14_21-51/test/test_3_forecasts_ch-1_v2.png
[EVAL 3/4] ECL | mamba | variate | L192_T96
       test mse : 0.136389
       npz      : runs_debug/ECL/variate/mamba/L192_T96/seed_42/2026-03-14_21-54/test/test_outputs.npz
       plot     : runs_debug/ECL/variate/mamba/L192_T96/seed_42/2026-03-14_21-54/test/test_3_forecasts_ch-1_v2.png
[EVAL 4/4] ECL | mamba | variate | L96_T96
       test mse : 0.149846
       npz      : runs_debug/ECL/variate/mamba/L

## later, full 24 for seed 42

In [23]:
base_cfg = build_base_config(seed=42, batch_size=32)

experiment_grid_24 = make_experiment_grid(
    experiment_modes=["multivariate", "increasing_lookback"],
    datasets=["ETTh1", "ECL"],
    models=["gru", "transformer", "mamba"],
    token_styles=["time", "variate"],
)


Seed 42 with NPZ + plots

In [27]:
bundle_seed42 = run_train_and_eval_loop(
    experiment_grid=experiment_grid_24,
    base_cfg=base_cfg,
    root_dir_runs="runs_debug",
    eval_save_npz_splits=("test",),
    eval_make_test_plot=True,
    eval_plot_num_random=3,
    eval_plot_seed=40,
    eval_plot_channel_idx=-1,
    eval_include_val=False,
    verbose_train=False,
    verbose_eval=False,
)



[1/24] mode=multivariate | ETTh1 | gru | time | L96_T96 | BS=32 | seed=42
TRAIN OK  | best_val=0.453388 | best_epoch=2
          saved: runs_debug/ETTh1/time/gru/L96_T96/seed_42/2026-03-14_22-44
EVAL  OK  | test mse=0.636536
          test dir: runs_debug/ETTh1/time/gru/L96_T96/seed_42/2026-03-14_22-44/test

[2/24] mode=multivariate | ETTh1 | gru | variate | L96_T96 | BS=32 | seed=42
TRAIN OK  | best_val=0.370653 | best_epoch=0
          saved: runs_debug/ETTh1/variate/gru/L96_T96/seed_42/2026-03-14_22-44
EVAL  OK  | test mse=0.437643
          test dir: runs_debug/ETTh1/variate/gru/L96_T96/seed_42/2026-03-14_22-44/test

[3/24] mode=multivariate | ETTh1 | transformer | time | L96_T96 | BS=32 | seed=42
TRAIN OK  | best_val=0.427718 | best_epoch=2
          saved: runs_debug/ETTh1/time/transformer/L96_T96/seed_42/2026-03-14_22-44
EVAL  OK  | test mse=0.553834
          test dir: runs_debug/ETTh1/time/transformer/L96_T96/seed_42/2026-03-14_22-44/test

[4/24] mode=multivariate | ETTh1 | t